In [1]:
import copy

import numpy as np
from aicspylibczi import CziFile
from pathlib import Path

import xml.etree.ElementTree as ET
import xml.dom.minidom

import cv2
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

from PIL import Image
from skimage import filters, segmentation, morphology, color, exposure, restoration, measure, feature
from skimage.measure import regionprops_table
from skimage.filters import threshold_otsu, threshold_triangle, threshold_local

from sklearn import preprocessing as p
from skimage.segmentation import find_boundaries
from skimage.measure import regionprops, label


from stardist.models import StarDist2D
from csbdeep.utils import normalize

from scipy import ndimage
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import tifffile

2025-05-13 12:37:43.712601: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-13 12:37:44.066915: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-13 12:37:44.370080: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747154264.633486 1263498 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747154264.708070 1263498 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-13 12:37:45.415642: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [2]:
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
# import scanpy as sc

# pip install kagglehub[pandas-datasets]
# import kagglehub


In [3]:
class HybridReprogramming():
    def __init__(self, sample, sampleID, basepath):
        self.sample = sample
        self.sampleID = sampleID
        self.channels = 3
        self.basepath = basepath

    def load_Genexpression(self):
        filepath = f'{self.basepath}/GeneExpression/Genexpression.h5ad'
        anndata = sc.read_h5ad(filepath)
        return anndata

    def load_frames(self, time_start, time_end):
        frames = []
        for t in range(time_start, time_end):
            filepath = f'{self.basepath}/Imaging/Imaging/{self.sample}/{self.sampleID}/t_{t+1}.hdf5'
            f = h5py.File(filepath, 'r')
            frame_shape = f['Scene'].shape
            frame = f['Scene'][:]
            frames.append(frame)
        framestack = np.stack(frames, axis=0)
        return framestack

In [7]:
def count_cells(sample, sampleid):
    path = '/scratch/indikar_root/indikar1/shared_data/HYB/kaggle_dataset/datasets/thedoodler/hybrid-imaging-and-genex-dataset-hyb-imagen/versions/3'
    reprogramming = HybridReprogramming(sample, sampleid, path)
    
    # TagGFP : Green
    # MKate : Red
    # Cy5 : Blue
    
    channelmap = {"TagGFP": 0, "MKate": 1, "Cy5": 2, "Oblique": 3}
    t1, t2 = 0, 135
    frames = reprogramming.load_frames(t1, t2)
    
    timepoints, channels, shapex, shapey = frames.shape
    
    mapshapex, mapshapey = shapex//6, shapey//5
    
    cell_counts = []
    for i in range(1, timepoints):
        print(f'running frame {i}')
        image = frames[i]
        nimage = plot_image(image)
        frame = Frame(nimage, i, plot=False)
    
        counters = frame.channel_runner()
        cell_counts.append(counters)
    
    basecountpath = '/nfs/turbo/umms-indikar/Ram/projects/reprogramming/Image_analysis/data'
    with open(f"{basecountpath}/{sample}_{sampleid}_counts.txt", "w") as file:
        for item in cell_counts:
            file.write(str(item) + "\n")
    print(sample, samplid, cell_counts)
    return cell_counts



### Run Cell Counts from Dataset

In [ ]:
# count_cells('Myod', 1)
count_cells('Myod', 2)
count_cells('PRRX1', 1)
count_cells('PRRX1', 2)
count_cells('Myod_PRRX1', 1)
count_cells('Myod_PRRX1', 2)
count_cells('Myod_PRRX1', 3)
count_cells('Negative_Controls', 1)
count_cells('Negative_Controls', 2)
count_cells('Negative_Controls', 3)